<a href="https://colab.research.google.com/github/lautarodibartolo-ae/t8001-pre-procesado-de-datos/blob/main/clase-1-tipos-y-formatos/01_tipos_y_formatos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Clase 1 — Los ejemplos

**Taller T8001 · Pre-procesado de datos**

Este notebook acompaña al apunte de la clase 1. No agrega temas nuevos: toma las ideas del apunte
que se entienden mejor viéndolas correr y las muestra funcionando, con la tabla de cuatro filas que
ya conocés. Cada bloque dice a qué sección del apunte corresponde.

Se lee el apunte primero y se ejecuta esto después.

## Cómo se usa

Antes de tocar nada: `Archivo` → `Guardar una copia en Drive`. Si no, los cambios se pierden.

Un notebook tiene dos tipos de celda. Las de texto, como esta, explican. Las de código ejecutan
Python y muestran el resultado justo abajo.

**Para ejecutar una celda: hacé clic adentro y apretá `Shift + Enter`.** El cursor pasa solo a la
siguiente.

Tres reglas:

1. Se ejecuta **en orden, de arriba hacia abajo**. Cada celda usa lo que dejaron las anteriores.
2. Si algo da un resultado raro, casi siempre es por haber ejecutado en desorden. Se arregla con
   `Entorno de ejecución` → `Reiniciar y ejecutar todo`.
3. **Una celda de este notebook falla a propósito.** Está avisada. Cuando llegues, seguí de largo.

Ojo con una consecuencia de las dos últimas: `Reiniciar y ejecutar todo` se frena en la celda que
falla. Después de usarlo, seguí a mano desde la celda siguiente al bloque rojo.

Los archivos que crea el notebook se ven en el panel de la izquierda, en el ícono de carpeta.

### Empezá por esta

Ejecutala con `Shift + Enter`. Si aparece un resultado abajo, ya está todo listo.

In [ ]:
print("Todo funciona.")
1 + 1

Dos cosas pasaron ahí. `print(...)` muestra lo que le pongas adentro. Y `1 + 1`, la última línea de
la celda, se muestra sola: el notebook siempre muestra el resultado de la última línea.

### Las herramientas

`pandas` es la librería que sabe abrir archivos de datos y trabajar con tablas. Se trae una sola
vez, al principio, y `as pd` le pone un apodo para no escribirla entera cada vez.

In [ ]:
import pandas as pd
from pathlib import Path

Path("crudo").mkdir(exist_ok=True)
Path("generado").mkdir(exist_ok=True)

print("pandas", pd.__version__)

Esas dos carpetas son la **regla del dato crudo intacto** del apunte, sección 3.8. En `crudo/` va lo
que llega de la fuente y solo se lee. En `generado/` va todo lo que produce el notebook, y se puede
borrar entera porque se reconstruye sola.

---
# 1. El CSV por dentro

*Apunte, sección 3.4.*

En vez de bajar un archivo de internet, lo fabricamos acá. Así el notebook funciona siempre, aunque
el portal del que bajaríamos el dato esté caído.

In [ ]:
tabla = '''id,localidad,personas,ambientes,fecha_visita,ingreso
1,Ramallo,3,2,2025-03-14,185000
2,Córdoba,5,3,2025-03-15,240500
3,Concepción,2,1,2025-03-15,
4,Ramallo,4,2,2025-03-16,198000
'''

Path("crudo/hogares.csv").write_text(tabla, encoding="utf-8")
print("archivo escrito")

Ahora miramos el archivo **antes** de que pandas lo interprete. Un CSV es texto plano: se puede leer
con los ojos.

In [ ]:
print(Path("crudo/hogares.csv").read_text(encoding="utf-8"))

Mirá la fila con `id` 3: termina en coma. Eso significa que el último valor, el ingreso, está
vacío. El encabezado también se imprime, así que esa es la cuarta línea de la salida.

---
# 2. Las cinco preguntas de la primera mirada

*Apunte, sección 4.2.*

Recién ahora se lo damos a pandas. `read_csv` devuelve una tabla, y a esa tabla la llamamos `df`.

In [ ]:
df = pd.read_csv("crudo/hogares.csv")
df

Las cinco preguntas, una por línea.

In [ ]:
print("1. tamaño:", df.shape)
print("2. columnas:", list(df.columns))
print()
print("3. tipos:")
print(df.dtypes)
print()
print("4. faltantes:")
print(df.isna().sum())
print()
print("5. duplicados:", df.duplicated().sum())

`df.shape` devuelve `(4, 6)`: cuatro filas y seis columnas. Es lo primero que se mira siempre, y en
el bloque 7 vas a ver por qué.

Si donde dice `str` te aparece `object`, es lo mismo: así llamaba pandas al texto en las versiones
anteriores.

Ojo con dos respuestas de la línea 3:

- **`id` es un número**, aunque sea una etiqueta. Sumar identificadores no significa nada.
- **`ingreso` es decimal y no entero**, justamente porque le falta un valor.

---
# 3. La herramienta ya decidió por vos

*Apunte, sección 4.3.*

`fecha_visita` se ve como una fecha, pero pandas la leyó como texto. Mientras siga siendo texto, no
se puede restar ni ordenar por mes.

In [ ]:
fechas = pd.to_datetime(df["fecha_visita"])

print("como la leyó pandas: ", df["fecha_visita"].dtype)
print("después de convertir:", fechas.dtype)
print("mes de cada visita:  ", list(fechas.dt.month))

df["fecha_visita"] = fechas

El otro caso del apunte son los identificadores que pierden el cero de adelante. Acá está, en dos
líneas.

In [ ]:
Path("crudo/codigos.csv").write_text("codigo\n0074\n0180\n", encoding="utf-8")

print("como lo lee pandas:", list(pd.read_csv("crudo/codigos.csv")["codigo"]))
print("como hay que leerlo:", list(pd.read_csv("crudo/codigos.csv", dtype=str)["codigo"]))

`0074` es un código de producto para vos. Para pandas es el número 74. `dtype=str` le dice que lo
deje como texto.

Y el tercer caso, el más frecuente en archivos reales: una ausencia escrita con palabras.

In [ ]:
Path("crudo/encuesta.csv").write_text("edad\n34\ns/d\n41\n", encoding="utf-8")

crudo = pd.read_csv("crudo/encuesta.csv")
print("tipo:     ", crudo["edad"].dtype, "  faltantes:", crudo["edad"].isna().sum())

limpio = pd.read_csv("crudo/encuesta.csv", na_values=["s/d"])
print("con aviso:", limpio["edad"].dtype, "faltantes:", limpio["edad"].isna().sum())

Sin avisar, la columna entera queda como texto y el conteo de faltantes da cero. Con
`na_values=["s/d"]`, pandas entiende que ese texto es una ausencia: la columna vuelve a ser
numérica y el faltante aparece.

---
# 4. Cinco formatos para la misma tabla

*Apunte, sección 5.1.*

Guardamos la misma tabla cinco veces y comparamos.

In [ ]:
df.to_csv("generado/hogares.csv", index=False)
df.to_excel("generado/hogares.xlsx", index=False)
df.to_json("generado/hogares.json", orient="records", date_format="iso", force_ascii=False)
df.to_parquet("generado/hogares.parquet", index=False)
df.to_csv("generado/hogares.csv.gz", index=False)

for nombre in ["hogares.csv", "hogares.xlsx", "hogares.json", "hogares.parquet", "hogares.csv.gz"]:
    ruta = Path("generado") / nombre
    print(nombre, "->", ruta.stat().st_size, "bytes")

Con cuatro filas los pesos no dicen mucho, y no es lo que importa. Lo que importa es qué se
conserva. Eso lo vemos ahora.

---
# 5. El tipo que se pierde

*Apunte, sección 5.2.*

`fecha_visita` es una fecha de verdad en `df`. La guardamos en CSV y en Parquet, y volvemos a leer
los dos archivos.

In [ ]:
print("antes de guardar:  ", df["fecha_visita"].dtype)
print("vuelto del CSV:    ", pd.read_csv("generado/hogares.csv")["fecha_visita"].dtype)
print("vuelto del Parquet:", pd.read_parquet("generado/hogares.parquet")["fecha_visita"].dtype)

Del CSV vuelve como texto: el trabajo de conversión se perdió. Del Parquet vuelve como fecha,
porque Parquet guarda el esquema junto a los datos.

De ahí la regla: **CSV para entregar, Parquet para trabajar.** Y si el destino final es CSV, el
código que reconstruye los tipos es parte del entregable.

---
# 6. Codificación de caracteres

*Apunte, sección 6.1.*

Escribimos la misma palabra en dos encodings y miramos los bytes que quedan en el disco.

In [ ]:
Path("generado/utf8.txt").write_text("tamaño", encoding="utf-8")
Path("generado/latin1.txt").write_text("tamaño", encoding="latin-1")

print("UTF-8:  ", Path("generado/utf8.txt").read_bytes())
print("Latin-1:", Path("generado/latin1.txt").read_bytes())

La `b` de adelante no es una letra del texto: avisa que lo que sigue son bytes y no caracteres.

`tama` y `o` son iguales en las dos: donde las tablas coinciden, no hay problema. La ñ es donde se
separan. En UTF-8 son dos bytes, `\xc3\xb1`. En Latin-1 es uno solo, `\xf1`.

## El error ruidoso

*Apunte, sección 6.2.*

> **Esta celda falla a propósito.** Va a salir un bloque rojo largo y el notebook se frena ahí. Eso
> es lo que queremos ver. Leé la última línea del error y seguí con la celda de abajo.

El archivo está en Latin-1 y no se lo decimos, así que Python supone UTF-8.

In [ ]:
Path("crudo/hogares_latin1.csv").write_text(tabla, encoding="latin-1")

pd.read_csv("crudo/hogares_latin1.csv")

La línea que importa es la última del bloque rojo:

```
'utf-8' codec can't decode byte 0xf3 in position ...: invalid continuation byte
```

Te dice tres cosas: qué encoding supuso, qué byte no supo interpretar y en qué posición estaba.

Acá el byte es `\xf3`, que en Latin-1 es la ó de `Córdoba`: es el primer carácter con acento del
archivo. En el apunte pasa lo mismo con la ñ, que en Latin-1 es `\xf1`.

Este error es el barato. Te frena antes de que hagas nada mal.

La forma de resolverlo: probar los candidatos y mirar cuál produce texto legible.

In [ ]:
bytes_crudos = Path("crudo/hogares_latin1.csv").read_bytes()

for enc in ["utf-8", "utf-8-sig", "latin-1", "cp1252"]:
    linea = bytes_crudos.decode(enc, errors="replace").splitlines()[3]
    print(f"{enc:10}", linea)

Los dos primeros dejan un `�` donde va la ó: `utf-8-sig` es UTF-8 con una regla extra, así que
falla igual. Los dos últimos escriben `Concepción`, y son indistinguibles entre sí, porque Latin-1
y CP1252 usan el mismo byte para las vocales acentuadas. Cuál de los dos es el correcto no se puede
decidir mirando el archivo. Se elige el más probable y se anota como supuesto.

## El error silencioso

*Apunte, sección 6.3.*

Ahora al revés: un archivo bien hecho en UTF-8, leído como si fuera Latin-1. Latin-1 tiene un
carácter para cada byte posible, así que nunca falla.

In [ ]:
mojibake = pd.read_csv("crudo/hogares.csv", encoding="latin-1")
print(list(mojibake["localidad"]))

Ningún error. Solo datos corrompidos: `CÃ³rdoba`, `ConcepciÃ³n`. Eso se llama **mojibake**, y si ves
una `Ã` donde debería haber un acento, ya sabés qué pasó.

El error ruidoso corta en el acto y no rompe nada. El silencioso no avisa, y aparece semanas después
en un informe.

**Probá esto:** cambiá `encoding="latin-1"` por `encoding="cp1252"` en la celda de arriba. Sale lo
mismo, y esa es justamente la razón por la que los dos son indistinguibles.

---
# 7. El CSV argentino

*Apunte, sección 6.4.*

Separador `;`, decimal `,`. Es lo que exporta Excel en castellano.

In [ ]:
argentino = '''id;localidad;personas;ingreso
1;Ramallo;3;185000,50
2;Córdoba;5;240500,75
'''
Path("crudo/hogares_ar.csv").write_text(argentino, encoding="cp1252")

mal = pd.read_csv("crudo/hogares_ar.csv", encoding="cp1252")
print("shape:", mal.shape)
print("columnas:", list(mal.columns))
mal

`shape` dice `(2, 1)`: **una** columna donde tenía que haber cuatro.

Lo que se ve a la izquierda no es una columna: es el nombre de cada fila. pandas buscó comas. En el
encabezado no encontró ninguna, así que las cuatro columnas se le convirtieron en un solo nombre.
En cada fila de datos encontró una, la del decimal, así que partió la línea en dos y usó la mitad
de la izquierda como etiqueta.

No hubo ningún error. Por eso `shape` es lo primero que se mira.

In [ ]:
bien = pd.read_csv("crudo/hogares_ar.csv", sep=";", decimal=",", encoding="cp1252")
print("shape:", bien.shape)
print("tipo de ingreso:", bien["ingreso"].dtype)
bien

Con `sep=";"` aparecen las cuatro columnas. Con `decimal=","` la columna `ingreso` es numérica y se
puede promediar. Con `encoding="cp1252"` la ó de `Córdoba` llega entera. Si te olvidás del segundo,
las columnas están bien, pero `ingreso` queda como texto.

**Probá esto:** sacale `decimal=","` a la celda de arriba y volvé a ejecutarla. Mirá qué le pasa al
tipo de `ingreso`.

---
# 8. Una imagen es una grilla de números

*Apunte, sección 7.1.*

Fabricamos una imagen chica y la miramos como lo que es: números.

In [ ]:
from PIL import Image     # Pillow, la librería de imágenes
import random

random.seed(0)                      # para que el ruido salga igual siempre
img = Image.new("RGB", (200, 120))  # una imagen vacía de 200 por 120

for x in range(200):                # recorre las columnas
    for y in range(120):            # y dentro de cada una, las filas
        ruido = random.randint(-30, 30)
        rojo, verde, azul = x, min(255, y * 2), 128 + ruido
        img.putpixel((x, y), (rojo, verde, azul))

img.save("generado/prueba.png")
img

Ahora la miramos como lo que es: números.

In [ ]:
print("tamaño en píxeles:", img.size)
print("color del píxel (10, 10):", img.getpixel((10, 10)))
print("peso sin comprimir:", 200 * 120 * 3, "bytes")

Mirá el píxel (10, 10): el rojo vale 10, que es su x, y el verde vale 20, que es su y por dos.

Cada píxel son tres números de 0 a 255, uno por canal: rojo, verde y azul. Los pusimos a propósito
en función de la posición, así que se puede verificar.

El canal azul lleva un poco de ruido, también a propósito. Una foto real nunca es lisa, y esa
aspereza es justo lo que hace difícil comprimirla.

## Compresión con y sin pérdida

*Apunte, sección 7.4.*

La misma imagen, guardada en PNG (sin pérdida) y en JPEG (con pérdida).

In [ ]:
img.save("generado/prueba.jpg", quality=60)

crudo_bytes = 200 * 120 * 3
print("sin comprimir", crudo_bytes, "bytes")

for nombre in ["prueba.png", "prueba.jpg"]:
    peso = (Path("generado") / nombre).stat().st_size
    print(nombre, peso, "bytes,", round(crudo_bytes / peso, 1), "veces más chico")

Los dos pesan menos que los 72000 bytes del crudo, pero no en la misma medida. El PNG tiene que
guardar cada píxel exacto, ruido incluido, y por eso baja poco. El JPEG tira parte de ese ruido y
por eso baja muchísimo más.

La pregunta es qué se paga por esa diferencia. La celda que sigue la contesta.

In [ ]:
png = Image.open("generado/prueba.png")
jpg = Image.open("generado/prueba.jpg")

print("píxel (10, 10) en el original:", img.getpixel((10, 10)))
print("píxel (10, 10) desde el PNG:  ", png.getpixel((10, 10)))
print("píxel (10, 10) desde el JPEG: ", jpg.getpixel((10, 10)))

El PNG devuelve el mismo píxel. El JPEG devuelve uno cerca, pero no igual: mirá el tercer número,
el azul. Esa diferencia no vuelve nunca más. Por eso el original se guarda sin pérdida y las
versiones comprimidas se generan a partir de él, no al revés.

**Probá esto:** cambiá `quality=60` por `quality=10` en la celda de más arriba, ejecutá de nuevo
esa celda y esta, y mirá cuánto se aleja el píxel.

---
# 9. Abrir un archivo propio

*Apunte, sección 3.6.*

Los ocho bloques anteriores trabajaron sobre archivos que fabricó el notebook. Este hace el camino
entero sobre una planilla desprolija, la de la sección 3.6 del apunte: título arriba, una fila en
blanco, encabezado, datos, una fila de totales y un faltante escrito `s/d`.

Primero la fabricamos, para no depender de que nadie nos mande nada.

In [ ]:
from openpyxl import Workbook   # la librería que escribe archivos de Excel

libro = Workbook()
hoja = libro.active
hoja.title = "Relevamiento"

hoja.append(["Relevamiento de hogares 2025 - Municipio de Ramallo"])
hoja.append([])
hoja.append(["id", "localidad", "personas", "ambientes", "fecha_visita", "ingreso"])
hoja.append([1, "Ramallo", 3, 2, "2025-03-14", 185000])
hoja.append([2, "Córdoba", 5, 3, "2025-03-15", 240500])
hoja.append([3, "Concepción", 2, 1, "2025-03-15", "s/d"])
hoja.append([4, "Ramallo", 4, 2, "2025-03-16", 198000])
hoja.append([None, "TOTAL", 14, 8, None, 623500])

libro.save("crudo/relevamiento.xlsx")
print("archivo escrito")

Ahora la leemos sin avisar nada, como veníamos leyendo todo. `read_excel` es el `read_csv` de los
archivos de Excel.

In [ ]:
crudo = pd.read_excel("crudo/relevamiento.xlsx")

print("shape:", crudo.shape)
print("columnas:", list(crudo.columns))
crudo

Ahí está la planilla entera, tal cual la ve pandas. Mirá tres cosas.

`shape` dice siete filas, y hogares hay cuatro: la fila en blanco, la del encabezado y la de
totales entraron como datos. El título se convirtió en el nombre de la primera columna, y las otras
cinco quedaron sin nombre. Y la columna `ingreso` no existe todavía: lo que hay es una columna
`Unnamed: 5` de texto, con la palabra `s/d` adentro.

Las cinco preguntas sobre esta tabla no contestan nada. Falta decirle tres cosas al leer.

In [ ]:
pd.ExcelFile("crudo/relevamiento.xlsx").sheet_names

La primera es **qué hoja**. Un `.xlsx` puede tener varias y `read_excel` lee la primera si no le
aclarás. Esta tiene una sola, y se llama `Relevamiento`.

Las otras dos son en qué fila empieza el encabezado y qué texto significa ausencia.

In [ ]:
hogares = pd.read_excel(
    "crudo/relevamiento.xlsx",
    sheet_name="Relevamiento",
    header=2,              # el encabezado está en la fila 3 de Excel, y pandas cuenta desde 0
    na_values=["s/d"],     # este texto es una ausencia, no un dato
)
hogares

Seis columnas con sus nombres de verdad. Las dos filas de arriba del encabezado desaparecieron
solas: `header=2` las saltea. Y donde decía `s/d` ahora dice `NaN`, que es como pandas escribe un
faltante.

Queda la fila de totales, y esa no la saca ningún parámetro.

In [ ]:
datos = hogares[hogares["localidad"] != "TOTAL"]

print("con la fila de totales: ", len(hogares), "filas, promedio", hogares["personas"].mean())
print("sin la fila de totales:", len(datos), "filas, promedio", datos["personas"].mean())

3,5 personas por hogar es el promedio de verdad. 5,6 es lo que da cuando el total se cuenta como si
fuera un hogar más, y nadie avisa.

Se saca por contenido, y para eso hay que haber mirado el archivo antes: acá sabemos que la fila de
totales dice `TOTAL` en la columna `localidad`. La tabla sin esa fila queda con el nombre `datos`.

Recién ahora las cinco preguntas contestan sobre los datos.

In [ ]:
print("1. tamaño:", datos.shape)
print("2. columnas:", list(datos.columns))
print()
print("3. tipos:")
print(datos.dtypes)
print()
print("4. faltantes:")
print(datos.isna().sum())
print()
print("5. duplicados:", datos.duplicated().sum())

Las mismas respuestas del bloque 2, sobre un archivo que no venía listo: cuatro filas, seis
columnas, un faltante en `ingreso`, ningún duplicado.

Con una marca de lo que pasó. `id` volvió como decimal, y en el bloque 2 era entero. La culpa es de
la fila de totales, que tenía el `id` vacío: pandas eligió el tipo cuando leyó el archivo, con esa
fila adentro, y sacarla después no lo deshace. `fecha_visita` sigue siendo texto, como siempre.

## Traer tu propio archivo

Colab corre en una máquina de Google, así que el archivo tiene que llegar hasta ahí. Abrí el panel
de la izquierda, el ícono de carpeta, y arrastrá el archivo adentro. Cuando termina de subir
aparece en la lista, y ya se lee con su nombre.

Dura lo que dura la sesión: si el entorno se reinicia, hay que subirlo otra vez.

Cuando Python contesta `FileNotFoundError`, esta celda dice por qué. Muestra dónde está parado el
notebook y qué archivos ve.

In [ ]:
print("estoy parado en:", Path.cwd())
print()
for ruta in sorted(Path.cwd().iterdir()):
    print(" ", ruta.name)

Tu lista va a ser distinta de la de esta salida, y eso está bien. Lo que importa es que el archivo
que subiste aparezca ahí con el nombre exacto, mayúsculas y acentos incluidos.

**Probá esto:** subí una planilla tuya, cambiá el nombre del archivo en la celda de `read_excel` y
volvé a ejecutar desde ahí. Vas a tener que ajustar `header=` según en qué fila esté tu encabezado,
y `na_values=` según con qué texto marquen los faltantes. Si el archivo es un CSV, cambiá
`read_excel` por `read_csv`: `header=` y `na_values=` funcionan igual.

---
# Cierre

Los nueve bloques que ejecutaste:

1. Un CSV es texto plano, y conviene mirarlo antes de que la herramienta lo interprete.
2. Las cinco preguntas de la primera mirada, sobre una tabla de verdad.
3. La herramienta adivina los tipos, y cuando se equivoca no avisa.
4. Cinco formatos para el mismo dato.
5. El tipo que se pierde entre CSV y Parquet.
6. El mismo texto en dos encodings, el error que corta y el que no.
7. El CSV argentino, que se lee mal sin dar ningún error.
8. Una imagen como grilla de números, y qué se pierde al comprimirla.
9. Una planilla de Excel desprolija, leída mal y leída bien, hasta las cinco preguntas.

Quedan afuera del notebook, porque se entienden leyendo, la regla cardinal, las siete dimensiones
de calidad, lo que Excel cambia sin preguntar, el sonido, el video y los metadatos EXIF. Todo eso está en el apunte.

Y si querés seguir probando: las cuatro celdas marcadas con **Probá esto** son un buen lugar para
empezar. Lo que produce el notebook está en `generado/` y se puede borrar entero. Para reconstruir
todo, `Entorno de ejecución` → `Reiniciar y ejecutar todo`, y después seguí a mano desde la celda
que viene atrás del error de encoding, porque ahí se frena.